# Part 09. 프로젝트 루트 확인과 4개 CSV 불러오기

 

## 35. 프로젝트 루트 설정

In [2]:

from pathlib import Path
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
data_dir = project_root / "data" / "raw"
print("프로젝트 루트:", project_root)
print("데이터 폴더:", data_dir)
print("데이터 폴더 존재:", data_dir.exists())

프로젝트 루트: c:\dev\ai-data-analysis2
데이터 폴더: c:\dev\ai-data-analysis2\data\raw
데이터 폴더 존재: True


## 36. pandas와 CSV 불러오기

In [3]:

import pandas as pd

customers = pd.read_csv(data_dir / "customers.csv")
products = pd.read_csv(data_dir / "products.csv")
orders = pd.read_csv(data_dir / "orders.csv")
order_items = pd.read_csv(data_dir / "order_items.csv")

## 37. 기본 구조와 주요 키 확인

In [4]:
datasets = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
}

for name, df in datasets.items():
    print(name, df.shape, df.columns.tolist())

customers (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products (100, 4) ['product_id', 'product_name', 'category', 'price']
orders (604, 5) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items (764, 5) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']


In [5]:
key_checks = {
    "customers.customer_id": customers["customer_id"],
    "products.product_id": products["product_id"],
    "orders.order_id": orders["order_id"],
    "order_items.order_item_id": order_items["order_item_id"],
}

for name, series in key_checks.items():
    print(
        name,
        "결측:", series.isna().sum(),
        "중복:", series.duplicated().sum(),
    )
    # 병합 실습 전에 기준 테이블의 주요 키가 고유한지 확인합니다.
    # isna()는 결측값 여부를 확인하는 메서드입니다. sum()을 사용하면 True 값의 개수를 셀 수 있습니다.
    # duplicated()는 중복 여부를 확인하는 메서드입니다. sum()을 사용하면 True 값의 개수를 셀 수 있습니다.
    

customers.customer_id 결측: 0 중복: 0
products.product_id 결측: 0 중복: 0
orders.order_id 결측: 0 중복: 300
order_items.order_item_id 결측: 0 중복: 0


# Part 10. 컬럼 선택·조건 필터링·정렬

 

## 38. Series와 DataFrame 선택

In [6]:
city_series = customers["city"]
customer_view = customers[
    ["customer_id", "gender", "age", "city"]
]

print(type(city_series))
print(type(customer_view))
display(customer_view.head())

<class 'pandas.Series'>
<class 'pandas.DataFrame'>


,customer_id,gender,age,city
0,1,F,19,광주
1,2,F,32,대구
2,3,F,61,성남
3,4,F,55,울산
4,5,F,19,부산


In [7]:
# 39. 단일 조건 필터링


customers_over_30 = customers[
    customers["age"] >= 30
]
#나이의 값이 30 이상인 값만 df로 가져와라
print(len(customers), len(customers_over_30))
display(customers_over_30.head())
#전체 customers의 수와 30세 이상인 고객 수를 출력하고, 30세 이상인 고객의 상위 5개 행을 표시합니다.


150 111


,customer_id,name,gender,age,city,signup_date
1,2,김정호,F,32,대구,2025-11-30
2,3,이경수,F,61,성남,2024-07-10
3,4,조영호,F,55,울산,2026-05-11
5,6,김지원,F,32,성남,2026-07-25
6,7,이상현,F,53,인천,2025-01-09


In [8]:
#40. 복합 조건 필터링

#30세 이상이면서 서울 거주:

seoul_over_30 = customers[
    (customers["age"] >= 30)
    & (customers["city"] == "서울")
]

display(seoul_over_30.head())

#서울 또는 부산:

seoul_or_busan = customers[
    customers["city"].isin(["서울", "부산"])
]
display(
    seoul_or_busan["city"].value_counts()
)
#isin()은 특정 값이 시리즈에 포함되어 있는지 확인하는 메서드입니다. value_counts()는 각 고유 값의 개수를 세는 메서드입니다.

#완료 주문이 아닌 주문:

not_completed = orders[
    ~(orders["order_status"] == "completed")
]
#~는 not 연산자입니다. '이것이 아니다'라는 말=여집합. 괄호 안의 조건이 False인 행만 가져옵니다. 여기서는 cancelled와 refunded를 가져옵니다.

display(
    not_completed["order_status"].value_counts(
        dropna=False
    )
)
#value_counts()는 각 고유 값의 개수를 세는 메서드입니다. dropna=False를 지정하면 결측값도 개수에 포함됩니다.

,customer_id,name,gender,age,city,signup_date
8,9,송지민,M,69,서울,2025-11-16
14,15,장정식,M,69,서울,2026-07-02
29,30,이민재,F,32,서울,2023-08-11
47,48,김예은,F,47,서울,2025-04-29
65,66,김재호,F,39,서울,2025-12-31


city
부산    16
서울    15
Name: count, dtype: int64

order_status
cancelled    129
refunded     104
NaN            3
Name: count, dtype: int64

In [9]:
#41. 상품 가격 정렬

 
expensive_products = (
    products
    .sort_values("price", ascending=False)
    .head(10)
)
#sort_values()는 특정 열을 기준으로 정렬하는 메서드입니다. ascending=False를 지정하면 내림차순으로 정렬됩니다. head(10)을 사용하면 상위 10개의 행만 가져옵니다.
display(
    expensive_products[
        [
            "product_id",
            "product_name",
            "category",
            "price",
        ]
    ]
)

,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
57,58,식품 상품 058,식품,197000
42,43,뷰티 상품 043,뷰티,197000
23,24,스포츠 상품 024,스포츠,196000
8,9,스포츠 상품 009,스포츠,193000
36,37,뷰티 상품 037,뷰티,193000
71,72,뷰티 상품 072,뷰티,189000
7,8,스포츠 상품 008,스포츠,189000
52,53,생활용품 상품 053,생활용품,188000


# Part 11. line_total 생성과 전체 주문 금액 구분

In [10]:
#42. 작업용 복사본과 파생 컬럼


order_items_work = order_items.copy()
#데이터 프레임에 새로운 컬럼을 추가할 때는 원본 데이터프레임을 변경하지 않고 작업용으로 복사본을 만들어 사용하는 것이 좋습니다. copy() 메서드를 사용하면 데이터프레임을 복사할 수 있습니다.
order_items_work["line_total"] = (
    order_items_work["quantity"]
    * order_items_work["unit_price"]
)
#line_total 이라는 새로운 칼럼을 만든다. 수량과 단위가격을 곱한 값.

#결과 확인:

display(
    order_items_work[
        [
            "order_item_id",
            "order_id",
            "product_id",
            "quantity",
            "unit_price",
            "line_total",
        ]
    ].head()
)

,order_item_id,order_id,product_id,quantity,unit_price,line_total
0,1,1,100,3,102000,306000
1,2,1,87,5,25000,125000
2,3,1,7,3,142000,426000
3,4,1,9,3,193000,579000
4,5,2,72,4,189000,756000


In [11]:
#43. 수작업 검증

 

sample = order_items_work.iloc[0]
#iloc[0]은 데이터프레임의 첫 번째 행을 선택하는 방법입니다. sample 변수에 첫 번째 행의 데이터를 저장합니다.

expected = sample["quantity"] * sample["unit_price"]
actual = sample["line_total"]
print("수작업:", expected)
print("파생 컬럼:", actual)
print("일치:", expected == actual)

수작업: 306000
파생 컬럼: 306000
일치: True


In [12]:
# 44. 전체 주문상세 금액

all_order_amount = order_items_work["line_total"].sum()

print("전체 주문상세 금액:", all_order_amount)

전체 주문상세 금액: 255610000


# Part 12. 주문 데이터 병합과 완료 주문 분석셋 만들기

In [13]:
# 45. 병합용 주문 컬럼 선택

orders_for_merge = orders[
    [
        "order_id",
        "customer_id",
        "order_date",
        "order_status",
    ]

].copy()

print(orders_for_merge.shape)
print(orders_for_merge.head())

(604, 4)
       order_id  customer_id  order_date order_status
0  <<<<<<< HEAD          NaN         NaN          NaN
1             1        123.0  2026-06-04    completed
2             2         77.0  2025-08-20    cancelled
3             3        138.0  2025-12-17    cancelled
4             4         57.0  2026-02-27    cancelled


In [14]:
# 46. 주문상세와 주문 병합

order_sales = (
    order_items_work
    .merge(
        orders_for_merge,
        on="order_id",
        how="left",
        validate="many_to_one",
        indicator="order_match",
    )

)

ValueError: You are trying to merge on int64 and str columns for key 'order_id'. If you wish to proceed you should use pd.concat

In [ ]:
# 47. 병합 검증

print("병합 전 행 수:", len(order_items_work))
print("병합 후 행 수:", len(order_sales))
display(
    order_sales["order_match"].value_counts(
        dropna=False
    )
)
# 여기선 병합해도 행 수는 같지만, 칼럼의 수는 다를 것이다. 

병합 전 행 수: 764
병합 후 행 수: 764


order_match
both          764
left_only       0
right_only      0
Name: count, dtype: int64

In [ ]:
# 미매칭 확인:

unmatched_orders = order_sales[
    order_sales["order_match"] != "both"
]
display(unmatched_orders.head())
# 아래 아무것도 안 나오기 때문에 미매칭이 없다는 것 확인


,order_item_id,order_id,product_id,quantity,unit_price,line_total,customer_id,order_date,order_status,order_match


In [ ]:
# 48. 완료 주문 분석셋

display(
    order_sales["order_status"].value_counts(
        dropna=False
    )
)
# VALUE_COUNTS()는 각 고유 값의 개수를 세는 메서드입니다. dropna=False를 지정하면 결측값도 개수에 포함됩니다.

order_status
completed    474
cancelled    162
refunded     128
Name: count, dtype: int64

In [ ]:
completed_sales = order_sales[
    order_sales["order_status"] == "completed"
].copy()
# completed에 대해서면 값을 가져오려고 한다. copy해서 데이터프레임 복사하고 뽑아냄. 여기선 print를 안 했기 때문에 값이 나오진 않음


In [ ]:
print("완료 주문상세 행:", len(completed_sales))
print(
    "완료 주문 수:",
    completed_sales["order_id"].nunique(),
)
print(
    "완료 주문 고객 수:",
    completed_sales["customer_id"].nunique(),
)
print(
    "완료 주문 매출:",
    completed_sales["line_total"].sum(),
)
#nunique()는 고유 값의 개수를 세는 메서드입니다. sum()은 합계를 계산하는 메서드입니다.

완료 주문상세 행: 474
완료 주문 수: 184
완료 주문 고객 수: 100
완료 주문 매출: 148990000


# Part 13. 상품 데이터 병합과 카테고리·상품 매출

In [ ]:
# 49. 필요한 상품 정보만 선택

products_for_merge = products[
    [
        "product_id",
        "product_name",
        "category",
    ]
].copy()

print(products_for_merge.head())

   product_id product_name category
0           1  전자기기 상품 001     전자기기
1           2    도서 상품 002       도서
2           3  전자기기 상품 003     전자기기
3           4  생활용품 상품 004     생활용품
4           5    식품 상품 005       식품


In [ ]:
#50. 완료 주문상세와 상품 병합

completed_items = (
    completed_sales
    .merge(
        products_for_merge,
        on="product_id",
        how="left",
        validate="many_to_one",
        indicator="product_match",
    )
)

In [ ]:
print(len(completed_sales), len(completed_items))
display(
    completed_items["product_match"].value_counts(
        dropna=False
    )
)

474 474


product_match
both          474
left_only       0
right_only      0
Name: count, dtype: int64

In [ ]:
#51. 카테고리별 매출 - 여기서 '별'은 그룹. 'by'의 의미로, 카테고리별로 매출을 집계한다는 의미입니다.

category_sales = (
    completed_items
    .groupby("category", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
        quantity_sold=("quantity", "sum"),
        detail_row_count=("order_item_id", "count"),
    )
    .sort_values("total_sales", ascending=False)
)
# agg는 집계함수. 
display(category_sales)

,category,total_sales,order_count,customer_count,quantity_sold,detail_row_count
3,스포츠,31743000,85,67,295,100
5,전자기기,26400000,60,44,259,78
2,생활용품,23915000,65,50,272,83
1,뷰티,23383000,65,53,223,76
4,식품,16573000,36,31,133,42
0,도서,16389000,52,46,149,58
6,패션,10587000,33,27,111,37


In [ ]:
# 52. 카테고리 합계 검증

category_total = category_sales["total_sales"].sum()
completed_total = completed_items["line_total"].sum()
print(category_total)
print(completed_total)
print(category_total == completed_total)

148990000
148990000
True


In [ ]:

# 53. 상품별 매출

product_sales = (
    completed_items
    .groupby(
        ["product_id", "product_name", "category"],
        as_index=False,
    )
    .agg(
        total_sales=("line_total", "sum"),
        quantity_sold=("quantity", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
    )
    .sort_values("total_sales", ascending=False)
)
#nunique를 쓰면 같은 사람이 두 번 사면 두 명으로 집계한다. 
display(product_sales.head(10))

,product_id,product_name,category,total_sales,quantity_sold,order_count,customer_count
39,41,스포츠 상품 041,스포츠,5705000,35,12,11
11,12,식품 상품 012,식품,4375000,25,7,7
8,9,스포츠 상품 009,스포츠,3860000,20,6,5
70,72,뷰티 상품 072,뷰티,3780000,20,6,6
69,71,전자기기 상품 071,전자기기,3703000,23,5,5
66,68,스포츠 상품 068,스포츠,3640000,26,8,8
78,81,전자기기 상품 081,전자기기,3630000,22,6,6
10,11,패션 상품 011,패션,3565000,31,7,7
20,22,생활용품 상품 022,생활용품,3248000,29,8,8
86,89,생활용품 상품 089,생활용품,3090000,30,11,11


# Part 14. 월별 매출과 고객별 구매 금액

In [ ]:
# 54. 주문 날짜 변환과 주문 월 생성

completed_items["order_date"] = pd.to_datetime(
    completed_items["order_date"],
    errors="coerce",
)
# coerce를 쓰면 변환 실패 시 NaT로 처리한다. errors="raise"를 쓰면 변환 실패 시 에러를 발생시킨다. errors="ignore"를 쓰면 변환 실패 시 원래 값을 그대로 유지한다.
print(
    "날짜 변환 실패:",
    completed_items["order_date"].isna().sum(),
)

날짜 변환 실패: 0


In [ ]:
completed_items["order_month"] = (
    completed_items["order_date"]
    .dt.to_period("M")
    .astype("string")
)
#dt.to_period("M")는 날짜를 월 단위로 변환하는 메서드입니다. astype("string")은 Period 객체를 문자열로 변환하는 메서드입니다.
#연월별로 집계하려면 order_month라는 새로운 칼럼을 만들어야 한다.
completed_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 474 entries, 0 to 473
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   order_item_id  474 non-null    int64         
 1   order_id       474 non-null    int64         
 2   product_id     474 non-null    int64         
 3   quantity       474 non-null    int64         
 4   unit_price     474 non-null    int64         
 5   line_total     474 non-null    int64         
 6   customer_id    474 non-null    int64         
 7   order_date     474 non-null    datetime64[us]
 8   order_status   474 non-null    str           
 9   order_match    474 non-null    category      
 10  product_name   474 non-null    str           
 11  category       474 non-null    str           
 12  product_match  474 non-null    category      
 13  order_month    474 non-null    string        
dtypes: category(2), datetime64[us](1), int64(7), str(3), string(1)
memory usage: 45.8 KB


In [ ]:
# 55. 월별 매출

monthly_sales = (
    completed_items
    .groupby("order_month", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
        quantity_sold=("quantity", "sum"),
    )
    .sort_values("order_month")
)
#sort_values("order_month")를 쓰면 월별로 오름차순 정렬된다.
display(monthly_sales)
#뭔가 오류남

,order_month,total_sales,order_count,customer_count,quantity_sold
0,2025-08,5869000,8,8,52
1,2025-09,16147000,19,19,132
2,2025-10,12385000,15,15,120
3,2025-11,23550000,24,23,233
4,2025-12,9876000,13,13,99
5,2026-01,10851000,13,13,105
6,2026-02,16504000,21,20,150
7,2026-03,9885000,18,16,102
8,2026-04,15536000,17,16,157
9,2026-05,15310000,19,18,152


In [ ]:
# 56. 고객별 구매 금액

customer_sales = (
    completed_items
    .groupby("customer_id", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        quantity_sold=("quantity", "sum"),
    )
)

In [ ]:
display(customer_sales.head())

,customer_id,total_sales,order_count,quantity_sold
0,3,3178000,2,26
1,4,603000,1,6
2,5,2004000,2,13
3,6,1349000,2,14
4,7,1173000,1,9


In [ ]:
customer_sales = (
    completed_items
    .groupby("customer_id", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        quantity_sold=("quantity", "sum"),
    )
    .sort_values("total_sales", ascending=False)
)
#여기서 sort_values는 custmomer_value를 내림차순 하는 것이기 때문에 괄호 안에 들어가 있어야 한다. 아래 display 값은 그냥 출력명령어이므로 그냥 아래 써도 무방
display(customer_sales.head(10))

,customer_id,total_sales,order_count,quantity_sold
76,117,4100000,5,48
62,102,3996000,4,35
51,83,3880000,4,39
21,30,3590000,5,32
29,40,3523000,4,27
13,20,3191000,2,25
0,3,3178000,2,26
70,111,3153000,3,38
42,66,3093000,4,30
97,147,2990000,2,21


In [ ]:
# 57. 고객 속성 연결

#개인정보 최소화를 위해 이름은 제외합니다.

customer_attributes = customers[
    ["customer_id", "gender", "age", "city"]
].copy()

In [ ]:
customer_sales_detail = (
    customer_sales
    .merge(
        customer_attributes,
        on="customer_id",
        how="left",
        validate="one_to_one",
        indicator="customer_match",
    )
    .sort_values("total_sales", ascending=False)
)

In [ ]:

display(
    customer_sales_detail["customer_match"].value_counts(
        dropna=False
    )
)
display(customer_sales_detail.head(10))

customer_match
both          100
left_only       0
right_only      0
Name: count, dtype: int64

,customer_id,total_sales,order_count,quantity_sold,gender,age,city,customer_match
0,117,4100000,5,48,F,65,성남,both
1,102,3996000,4,35,M,60,고양,both
2,83,3880000,4,39,F,22,수원,both
3,30,3590000,5,32,F,32,서울,both
4,40,3523000,4,27,M,23,서울,both
5,20,3191000,2,25,F,20,인천,both
6,3,3178000,2,26,F,61,성남,both
7,111,3153000,3,38,F,41,광주,both
8,66,3093000,4,30,F,39,서울,both
9,147,2990000,2,21,M,19,부산,both


# Part 15. 결과 CSV 저장과 반복 점검 함수

In [ ]:
# 58. 결과 폴더 생성

output_dir = project_root / "reports" / "chapter04"
output_dir.mkdir(parents=True, exist_ok=True)
print(output_dir)
#mkdir은 디렉토리 만들라는 명령어. parents=True는 상위 디렉토리가 없으면 상위 디렉토리도 생성하라는 의미. exist_ok=True는 이미 디렉토리가 존재하면 에러를 발생시키지 않고 넘어가라는 의미.

c:\dev\ai-data-analysis\reports\chapter04


In [ ]:
# 59. 결과 파일 저장

outputs = {
    "category_sales.csv": category_sales,
    "product_sales.csv": product_sales,
    "monthly_sales.csv": monthly_sales,
    "customer_sales.csv": customer_sales_detail,
}
for file_name, df in outputs.items():
    output_path = output_dir / file_name
    df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
    )
    print(
        file_name,
        output_path.exists(),
        output_path.stat().st_size,
    )

category_sales.csv True 309
product_sales.csv True 4833
monthly_sales.csv True 415
customer_sales.csv True 3448


In [ ]:
# 60. 저장 결과 다시 읽기

saved_category_sales = pd.read_csv(
    output_dir / "category_sales.csv"
)
display(saved_category_sales.head())
print(saved_category_sales.shape)

,category,total_sales,order_count,customer_count,quantity_sold,detail_row_count
0,스포츠,31743000,85,67,295,100
1,전자기기,26400000,60,44,259,78
2,생활용품,23915000,65,50,272,83
3,뷰티,23383000,65,53,223,76
4,식품,16573000,36,31,133,42


(7, 6)


In [ ]:
# 61. 병합 점검 함수
 
def check_merge_result(
    *,
    name: str,
    left_rows: int,
    merged: pd.DataFrame,
    indicator_column: str,
) -> None:
    print(f"[{name}]")
    print("병합 전 행 수:", left_rows)
    print("병합 후 행 수:", len(merged))
    print(
        merged[indicator_column].value_counts(
            dropna=False
        )
    )

In [ ]:
# 위에 만든 함수 호출

check_merge_result(
    name="주문상세-주문",
    left_rows=len(order_items_work),
    merged=order_sales,
    indicator_column="order_match",
)

[주문상세-주문]
병합 전 행 수: 764
병합 후 행 수: 764
order_match
both          764
left_only       0
right_only      0
Name: count, dtype: int64


In [ ]:
# 62. 집계 합계 검증 함수
# check_total 이라는 함수
def check_total(
    *,
    name: str,
    source_total: float,
    summary_total: float,
) -> None:
    difference = source_total - summary_total
    print(f"[{name}]")
    print("원본 합계:", source_total)
    print("요약 합계:", summary_total)
    print("차이:", difference)

In [ ]:
check_merge_result(
    name="주문상세-주문",
    left_rows=len(order_items_work),
    merged=order_sales,
    indicator_column="order_match",
)

[주문상세-주문]
병합 전 행 수: 764
병합 후 행 수: 764
order_match
both          764
left_only       0
right_only      0
Name: count, dtype: int64


| 검증 항목 | 확인 내용 | 결과 |
| :--- | :--- | :--- |
| DataFrame | 실제 변수명과 같은가? | 적합 (O): `orders`, `order_items`, `products`, `customers` 변수 사용|
| 컬럼 | 실제 컬럼만 사용하는가? | 적합 (O): `order_id`, `quantity`, `unit_price`, `category` 등 실제 컬럼 사용 |
| 상태값 | completed 표기가 맞는가? | 적합 (O): `"completed"` 상태값 반영 |
| 계산식 | quantity × unit_price인가? | 적합 (O): `line_total` 계산식 적용 (첫 행 값: `306000`) |
| 분석 범위 | 완료 주문만 포함하는가? | 적합 (O): `order_status == "completed"` 필터링 (완료 상세 행: `474`) |
| 주문 수 | nunique()를 사용하는가? | 적합 (O): `nunique()` 적용 (완료 주문 수: `184`) |
| 병합 키 | 실제 관계와 맞는가? | 적합 (O): `order_id`, `product_id` 기준 `many_to_one` 관계 적용 |
| validate | many_to_one이 적용되었는가? | 적합 (O): `validate="many_to_one"` 옵션 적용 |
| indicator | 미매칭을 확인하는가? | 적합 (O): both (`764`/`474`), left_only (`0`), right_only (`0`) 확인 |
| 행 수 | 병합 전후를 비교하는가? | 적합 (O): 주문상세-주문 (`764` → `764`), 완료 주문상세-상품 (`474` → `474`) 비교 출력 |
| 합계 | 원본과 요약 합계를 비교하는가? | 적합 (O): 카테고리 합계(`148990000`) == 완료 주문 전체 합계(`148990000`) (`True`) |
| 개인정보 | 원본 고객 정보를 요구하지 않는가? | 적합 (O): 이름(`name`) 등 민감 정보 제외 및 `gender`, `age`, `city` 사용 |